# BrainPrint: run TRIBE v2 on the 3 demo pitches

This notebook replaces the **synthetic placeholders** in `legacy/tribe/cache/` with real TRIBE v2 predictions.

1. Go to **Runtime → Change runtime type → GPU** (a T4 is enough).
2. On Hugging Face, request access to the gated **Llama 3.2** model that TRIBE uses as its text encoder. If access is missing, the error in step 4 names the exact repo.
3. Run all cells. When the install cell finishes, restart the session once, then continue from step 3.

Output: 3 JSON files, downloaded as `tribe_cache.zip`. Unzip them into `legacy/tribe/cache/` in your clone and commit.

What the data is: TRIBE predicts fMRI-like responses for an *average subject*, so every value is a predicted attention/processing signal, not a measurement of any viewer.

In [ ]:
!pip install -q "tribev2 @ git+https://github.com/facebookresearch/tribev2.git" nilearn==0.11.1
# Restart the session after this cell finishes (Runtime → Restart session).

## Step 3: Hugging Face login and fetch BrainPrint

In [ ]:
from huggingface_hub import login
login()  # paste a token from an account with access to the gated Llama 3.2 model

!git clone -q https://github.com/shubham123sing/brainprint.git || (cd brainprint && git pull -q)
import sys; sys.path.insert(0, "brainprint/legacy/tribe")

## Step 4: Load TRIBE v2 once and run all 3 demos

In [ ]:
import run_tribe
from networks import load_atlas

model = run_tribe.load_model("./tribe_cache")   # downloads about 1 GB of weights the first time
atlas = load_atlas()
DEMOS = [
    ("pitch_clear",     "Crumb: bakery leftovers (clear pitch)"),
    ("pitch_jargon",    "Ledgerly v1 (jargon-heavy pitch)"),
    ("pitch_jargon_v2", "Ledgerly v2 (revised pitch)"),
]
for sid, title in DEMOS:
    run_tribe.run(model, f"brainprint/legacy/data/demo/{sid}.txt", sid, title,
                  f"brainprint/legacy/tribe/cache/{sid}.json", atlas=atlas)

## Step 5: Validate and download

In [ ]:
!cd brainprint && python legacy/tribe/validate.py
!cd brainprint/legacy/tribe/cache && zip -q ../../../tribe_cache.zip *.json
from google.colab import files
files.download("tribe_cache.zip")

## Optional: a quick plot of the 7 networks

In [ ]:
import json, matplotlib.pyplot as plt
for sid, title in DEMOS:
    d = json.load(open(f"brainprint/legacy/tribe/cache/{sid}.json"))
    t = [s["t"] for s in d["seconds"]]
    plt.figure(figsize=(10, 2.5)); plt.title(f"{title}: predicted network response ({d['source']})")
    for n in d["networks"]:
        plt.plot(t, [s["networks"][n] for s in d["seconds"]], label=n)
    plt.legend(fontsize=7, ncol=7); plt.xlabel("seconds"); plt.show()